In [ ]:
import pandas as pd
import sys
print(sys.executable)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

In [ ]:
df = pd.read_csv('data.csv')

In [ ]:
df["month"] = pd.to_datetime(df["Time"]).dt.to_period("M")
df["dia"] = pd.to_datetime(df["Time"]).dt.to_period("D")
df['Amount'] = df['Amount'].str.replace(r'[A-Za-z]+', '', regex=True).astype(float)
df['Fee'] = df['Fee'].str.replace(r'[A-Za-z]+', '', regex=True).astype(float)
df['Executed'] = df['Executed'].str.replace(r'[A-Za-z]+', '', regex=True).astype(float)
df['profit'] = df.apply(lambda row: row['Amount'] - row['Fee'] if row['Side'] == 'SELL' else row['Amount'], axis=1)
df=(df[df['month']>='2024-01'] ).sort_values(['Pair', 'Time']).reset_index(drop=True)



In [ ]:
def filtrar_trades(df, data_inicio, data_fim#, side='SELL'
                   ):
    return (
        df[
            df['dia'].between(data_inicio, data_fim) #&
           # (df['Side'] == side)
        ]
        .assign(
            igual_linha_acima=lambda d: (
                #(d['Time'] == d['Time'].shift()) &
                (d['Pair'] == d['Pair'].shift()) &
                (d['Side'] == d['Side'].shift()) &
                (d['Price'] == d['Price'].shift())
            )
        )
        .pipe(
            lambda d: d.groupby((~d['igual_linha_acima']).cumsum()).agg(
                Time=('Time', 'first'),
                Pair=('Pair', 'first'),
                Side=('Side', 'first'),
                dia=('dia', 'first'),
                Price=('Price', 'first'),
                Executed=('Executed', 'sum'),
                Amount=('Amount', 'sum'),
                profit=('profit', 'sum'),
            ).reset_index(drop=True)
        )
    )



In [ ]:
df['Pair'].unique()

In [ ]:
resultados = []

for moeda in df['Pair'].unique():
    df_moeda = df[df['Pair'].isin([moeda])]
    
    df_buy = filtrar_trades(df_moeda, '2024-01-01', '2026-07-14')
    
    if df_buy.empty:
        continue  # pula moedas sem transações no período
    
    df_buy['Lucro'] = (
        df_buy['profit'] - df_buy['profit'].shift(1)
    ).where(df_buy.index % 2 == 1)
    
    df_buy['diferenca'] = (
        df_buy['Executed'] - df_buy['Executed'].shift(1)
    ).where(df_buy.index % 2 == 1)
    
    df_completos = df_buy[
        (df_buy['diferenca'] > -0.01) & (df_buy['diferenca'] < 0.0001)
    ]
    
    df_completos = df_completos.sort_values(['Pair', 'Time']).reset_index(drop=True)
    
    resultados.append(df_completos)

# Empilha tudo no final
df_final = pd.concat(resultados, ignore_index=True)
df_final.sort_values(['Time', 'Pair']).reset_index(drop=True)

d

In [ ]:
df_new = df_buy[~df_buy.index.isin(df_completos.index.union(df_completos.index - 1))]
drop_columns = ['diferenca', 'Lucro']
df_new = df_new.drop(columns=drop_columns, errors='ignore')

# --- 2. Preço e quantidade de compra, ffill por par ---
df_new['preco_compra'] = df_new.groupby('Pair')['Price'].transform(
    lambda x: x.where(df_new['Side'] == 'BUY').ffill()
)
df_new['amount_compra'] = df_new.groupby('Pair')['Executed'].transform(
    lambda x: x.where(df_new['Side'] == 'BUY').ffill()
)
# --- 3. Diferença de preço simples (venda atual vs compra) ---
df_new['diferenca_preco'] = ((df_new['Price'] / df_new['preco_compra']) - 1) * 100


# --- 4. Identifica grupos: novo grupo a cada BUY ou troca de par ---
novo_grupo = (df_new['Side'] == 'BUY') | (df_new['Pair'] != df_new['Pair'].shift())
df_new['grupo'] = novo_grupo.cumsum()

df_new['executed_venda'] = df_new['Executed'].where(df_new['Side'] == 'SELL', 0)
df_new['amount_acumulado'] = df_new.groupby('grupo')['executed_venda'].cumsum()
df_new.loc[df_new['Side'] == 'BUY', 'amount_acumulado'] = 0

# --- 6. Preço médio ponderado de venda (valor vendido / qtd vendida) ---
df_new['valor_vendido'] = (df_new['Price'] * df_new['Executed']).where(df_new['Side'] == 'SELL', 0)
df_new['valor_vendido_acum'] = df_new.groupby('grupo')['valor_vendido'].cumsum()
df_new['qtd_vendida_acum'] = df_new['amount_acumulado']  # é a mesma coisa, evita recalcular
df_new['amount_buy'] = df_new.groupby('grupo')['Amount'].transform(
    lambda x: x.where(df_new['Side'] == 'BUY').ffill()
)
df_new['preco_venda_ponderado'] = df_new['valor_vendido_acum'] / df_new['qtd_vendida_acum']
drop_columns = ['grupo','executed_venda','diferenca_preco','amount_acumulado','valor_vendido','profit','preco_venda_ponderado','Executed','Price']
df_new = df_new.drop(columns=drop_columns, errors='ignore')

df_new = df_new.sort_values('Time')
df_resumo = df_new.drop_duplicates(subset=('Pair','amount_buy','preco_compra'), keep='last')
df_resumo

In [ ]:
ßßßß

In [ ]:


# Lucro em valor de cada venda individual
df_new['lucro_venda'] = ((df_new['Price'] - df_new['preco_compra']) * df_new['Executed']).where(df_new['Side'] == 'SELL', 0)

# Total acumulado geral (todas as vendas, todos os pares, sem resetar)
df_new['lucro_total_acumulado'] = df_new['lucro_venda'].cumsum()
# --- 8. Limpa colunas auxiliares que não precisa mais ver ---
df_new = df_new.drop(columns=['executed_venda', 'valor_vendido', 'valor_vendido_acum','grupo'], errors='ignore')
# Garante que está ordenado por tempo (caso não esteja)
df_new = df_new.sort_values('Time')

# Pega a última transação de cada Pair
df_ultima = df_new.groupby('Pair').tail(1)
df_ultima

In [ ]:
df_new

In [ ]:
df_new = df_buy[~df_buy.index.isin(df_completos.index.union(df_completos.index - 1))]
df_new['preco_compra'] = df_new.groupby('Pair')['Price'].transform(
    lambda x: x.where(df_new['Side'] == 'BUY').ffill()
)
drop_columns = ['ganho', 'preco_d', 'diferenca', 'Lucro']
df_new = df_new.drop(columns=drop_columns, errors='ignore')
df_new

In [ ]:
# Garante que está ordenado por tempo (caso não esteja)
df_new = df_new.sort_values('Time')

# Pega a última transação de cada Pair
df_ultima = df_new.groupby('Pair').tail(1)
df_ultima

In [ ]:
df_final = pd.concat([df_completos, df_ultima], ignore_index=True)
df_final